# Freight Rate Prediction & Explainable AI (XAI) Pipeline

This notebook provides a complete, structured end-to-end Machine Learning workflow to predict spot freight load rates:
1. **Initial Data Profiling & Pre-Modeling Diagnostics** (Data types, summary statistics, missing values, duplicates, outliers, cardinality)
2. **Exploratory Data Analysis (EDA)** (Visualizing distributions, equipment tier premiums, and distance correlation)
3. **Feature Engineering & Imputation** (Temporal extraction, categorical encoding, leakage-free median imputation)
4. **Time-Based Model Validation** (XGBoost Regressor evaluated on a forward October holdout set)
5. **Explainable AI (XAI)** (Global Feature Gain, SHAP Beeswarm summary, and local Waterfall prediction breakdowns)
6. **Final Predictions Export** (Retraining on full data and exporting `validation_predictions.csv` and `december-chart-inputs.csv`)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.impute import SimpleImputer

# Ensure output directory exists
os.makedirs('output', exist_ok=True)

# Set visual theme
sns.set_theme(style='whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8

## 1. Data Loading

In [ ]:
train_df = pd.read_csv('train-test.csv')
val_df = pd.read_csv('validation.csv')
dec_df = pd.read_csv('december-chart-inputs.csv')

print(f"Historical Training Set Shape: {train_df.shape}")
print(f"Validation Test Set Shape:     {val_df.shape}")
print(f"December Chart Input Shape:    {dec_df.shape}")

train_df.head()

## 2. Basic Data Inspection & Pre-Modeling Diagnostics

Before building models or creating plots, we perform fundamental data hygiene and quality audits:
- Schema, data types, and required conversions
- Summary statistics across numerical and categorical features
- Missing value audit across training and inference datasets
- Duplicate row and primary key verification
- Outlier detection and range plausibility (IQR method)
- Cardinality analysis for categorical attributes

### 2.1 Data Types & Conversion Check
We verify column data types to identify any features that need casting (e.g., `date` string to `datetime`, and categorical strings to `category` dtype).

In [ ]:
print("=== Training Dataset Data Types & Non-Null Counts ===")
train_df.info()

print("\n=== Data Type Conversion Requirements ===")
print("1. 'date': Currently 'object' -> Needs conversion to pd.to_datetime for temporal feature extraction.")
print("2. 'pickup', 'delivery', 'equipment': Currently 'object' -> Needs categorical casting for native XGBoost handling.")
print("3. 'distance', 'weight', 'market_index', 'quote_signal': Float/Numerical -> Appropriate for modeling.")

### 2.2 Summary Statistics (`describe`)
We inspect the central tendencies, spread, percentiles, and min/max ranges for continuous variables, as well as categorical frequencies.

In [ ]:
print("=== Numerical Feature Statistics (describe) ===")
display(train_df.describe().T)

print("\n=== Categorical Feature Overview ===")
display(train_df.describe(include=['O']).T)

### 2.3 Missing & Null Values Audit
We check for missing values across the training, validation, and December datasets to design a leakage-free imputation strategy.

In [ ]:
null_counts = train_df.isnull().sum()
null_pct = (train_df.isnull().sum() / len(train_df)) * 100

null_summary = pd.DataFrame({
    'Missing Count': null_counts,
    'Missing Percentage (%)': null_pct.round(2)
})
print("=== Training Set Missing Values ===")
display(null_summary[null_summary['Missing Count'] > 0])

print("\n=== Schema Comparison with December Input ===")
missing_in_dec = set(train_df.columns) - set(dec_df.columns)
print(f"Columns absent in December dataset: {missing_in_dec}")
print("-> Note: December input will require placeholder generation and median imputation from training distribution.")

### 2.4 Duplicate Records & Primary Key Verification
We verify whether any identical records or repeated `load_id` values exist.

In [ ]:
exact_duplicates = train_df.duplicated().sum()
id_duplicates = train_df['load_id'].duplicated().sum()

print(f"Exact Duplicate Rows: {exact_duplicates}")
print(f"Duplicate 'load_id' entries: {id_duplicates}")
if exact_duplicates == 0 and id_duplicates == 0:
    print("-> Data Integrity Check PASSED: Every load record is unique.")

### 2.5 Outlier & Physical Range Analysis (IQR Method)
We evaluate the statistical distribution bounds (Q1, Q3, and 1.5 * IQR) for numerical variables to ensure values represent plausible freight loads.

In [ ]:
print("=== Outlier Diagnostic using Interquartile Range (IQR) ===")
for col in ['distance', 'weight', 'posted_rate']:
    q1 = train_df[col].quantile(0.25)
    q3 = train_df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    outliers = train_df[(train_df[col] < lower_bound) | (train_df[col] > upper_bound)]
    print(f"[{col}]")
    print(f"  Range: {train_df[col].min():.1f} to {train_df[col].max():.1f} | Median: {train_df[col].median():.1f}")
    print(f"  IQR Bounds: [{lower_bound:.1f}, {upper_bound:.1f}]")
    print(f"  Values outside 1.5*IQR: {len(outliers):,} rows ({len(outliers)/len(train_df)*100:.2f}%)\n")

print("-> Insight: Freight rates naturally skew upward on long-haul/heavy loads. Tree-based models (XGBoost) naturally handle these without arbitrary clipping.")

### 2.6 Categorical Cardinality & Unique Values
We examine the unique values and categories present in pickup/delivery hubs and equipment types.

In [ ]:
print("=== Categorical Cardinality ===")
print(f"Unique Pickup Hubs:   {train_df['pickup'].nunique()}")
print(f"Unique Delivery Hubs: {train_df['delivery'].nunique()}")
print(f"Unique Equipment:     {train_df['equipment'].nunique()} -> {train_df['equipment'].unique().tolist()}")

print("\nTop 5 Most Frequent Pickup Cities:")
print(train_df['pickup'].value_counts().head())

## 3. Exploratory Data Analysis (EDA)
With data quality verified, we explore the visual relationships and distributions driving freight rate formation.

### 3.1 Target Variable (`posted_rate`) Distribution

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(train_df['posted_rate'], bins=50, kde=True, color='#2b5c8f')
plt.title('Distribution of Posted Rates ($)', fontsize=13, pad=10)
plt.xlabel('Posted Rate ($)', fontsize=11)
plt.ylabel('Frequency', fontsize=11)
plt.tight_layout()
plt.savefig('output/eda_distribution.png', dpi=300)
plt.show()

### 3.2 Rate Variation by Equipment Type
Different truck equipment types command distinct baseline market rates due to refrigeration (Reefer) and handling requirements (Flatbed).

In [ ]:
plt.figure(figsize=(8, 5))
palette = {'Dry Van': '#336699', 'Flatbed': '#E67E22', 'Reefer': '#27AE60'}
sns.boxplot(x='equipment', y='posted_rate', data=train_df, palette=palette)
plt.title('Posted Rate by Equipment Type', fontsize=13, pad=10)
plt.xlabel('Equipment Type', fontsize=11)
plt.ylabel('Posted Rate ($)', fontsize=11)
plt.tight_layout()
plt.savefig('output/eda_equipment.png', dpi=300)
plt.show()

### 3.3 Distance vs. Posted Rate Correlation
Distance is the primary fundamental driver of total freight rate.

In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(x='distance', y='posted_rate', hue='equipment', alpha=0.35, data=train_df, palette=palette)
plt.title('Distance vs. Posted Rate by Equipment', fontsize=13, pad=10)
plt.xlabel('Distance (miles)', fontsize=11)
plt.ylabel('Posted Rate ($)', fontsize=11)
plt.tight_layout()
plt.savefig('output/eda_distance.png', dpi=300)
plt.show()

## 4. Feature Engineering & Preprocessing
We extract temporal calendar signals (`month`, `day_of_week`, `day_of_month`), cast categorical features, and align inference dataset schemas.

In [ ]:
def engineer_features(df, is_december=False):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    df['month'] = df['date'].dt.month
    df['day_of_week'] = df['date'].dt.dayofweek
    df['day_of_month'] = df['date'].dt.day
    
    # Handle missing columns for December test inputs
    if is_december:
        for col in ['market_index', 'quote_signal', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon']:
            df[col] = np.nan
            
    cat_cols = ['pickup', 'delivery', 'equipment']
    for col in cat_cols:
        df[col] = df[col].astype('category')
        
    return df

train_feat = engineer_features(train_df)
val_feat = engineer_features(val_df)
dec_feat = engineer_features(dec_df, is_december=True)
print("Feature engineering complete!")

## 5. Imputation & Time-Based Validation Split

To prevent temporal lookahead bias, we split data chronologically:
- **Training partition:** Jan 1, 2025 – Sep 30, 2025 (43,147 loads)
- **Validation holdout:** Oct 1, 2025 – Oct 31, 2025 (4,853 loads)
- Imputer is fit strictly on training data to prevent leakage.

In [ ]:
features = [
    'pickup', 'delivery', 'distance', 'equipment', 'weight',
    'market_index', 'quote_signal', 
    'month', 'day_of_week', 'day_of_month',
    'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon'
]
target = 'posted_rate'

num_cols = ['weight', 'market_index', 'quote_signal', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon']
imputer = SimpleImputer(strategy='median')
imputer.fit(train_feat[num_cols])

train_feat[num_cols] = imputer.transform(train_feat[num_cols])
val_feat[num_cols] = imputer.transform(val_feat[num_cols])
dec_feat[num_cols] = imputer.transform(dec_feat[num_cols])

# Time-based split
train_split = train_feat[train_feat['date'] < '2025-10-01']
val_split = train_feat[train_feat['date'] >= '2025-10-01']

X_train_split, y_train_split = train_split[features], train_split[target]
X_val_split, y_val_split = val_split[features], val_split[target]

print(f"Training split:   {X_train_split.shape[0]:,} loads")
print(f"Validation split: {X_val_split.shape[0]:,} loads")

## 6. Model Training & Holdout Validation
We train an `XGBoost Regressor` with native categorical partitioning and evaluate generalization error on the October holdout.

In [ ]:
model = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42
)

model.fit(X_train_split, y_train_split, eval_set=[(X_val_split, y_val_split)], verbose=50)

val_preds = model.predict(X_val_split)
val_rmse = np.sqrt(mean_squared_error(y_val_split, val_preds))
val_mae = mean_absolute_error(y_val_split, val_preds)

print(f"\n--- Validation Metrics (October Holdout) ---")
print(f"Validation RMSE: ${val_rmse:.2f}")
print(f"Validation MAE:  ${val_mae:.2f}")

## 7. Explainable AI (XAI) & Interpretability
We implement three XAI techniques to explain model decision logic:
1. **Global Feature Gain Importance** (Loss reduction per feature)
2. **SHAP Global Beeswarm Summary** (Directional impact of feature values)
3. **SHAP Local Waterfall Breakdown** (Decomposition of single load predictions)

In [ ]:
# 7.1 Global Feature Importance (Gain Metric)
importance = model.get_booster().get_score(importance_type='gain')
importance_df = pd.DataFrame({
    'Feature': list(importance.keys()),
    'Gain': list(importance.values())
}).sort_values(by='Gain', ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Gain'], color='#2b5c8f')
plt.title('XAI: Global Feature Importance (Average Gain)', fontsize=13, pad=10)
plt.xlabel('Gain (Improvement in Loss)', fontsize=11)
plt.tight_layout()
plt.savefig('output/xai_feature_importance.png', dpi=300)
plt.show()

In [ ]:
# 7.2 SHAP Global Feature Impact (Summary / Beeswarm Plot)
# Using native XGBoost TreeSHAP (fast and robust across versions)
sample_val = X_val_split.sample(n=min(1000, len(X_val_split)), random_state=42)
dmat_sample = xgb.DMatrix(sample_val, enable_categorical=True)
contribs = model.get_booster().predict(dmat_sample, pred_contribs=True)

shap_values = contribs[:, :-1]
base_values = contribs[:, -1]

shap_explanation = shap.Explanation(
    values=shap_values,
    base_values=base_values,
    data=sample_val,
    feature_names=features
)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_val, show=False)
plt.title('XAI: SHAP Summary (Directional Feature Impact)', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('output/xai_shap_summary.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 7.3 SHAP Local Explanation (Waterfall Plot for a Single Load)
plt.figure(figsize=(10, 6))
shap.plots.waterfall(shap_explanation[0], show=False)
plt.title('XAI: SHAP Local Prediction Breakdown (Sample Load)', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig('output/xai_shap_waterfall.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Final Retraining & Predictions Export
We retrain the model across all historical data (Jan – Oct) to capture full market recency, then generate the final forecasts.

In [ ]:
print('Retraining on full training data (48,000 loads)...')
model_full = xgb.XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    enable_categorical=True,
    random_state=42
)
model_full.fit(train_feat[features], train_feat[target])

print('Predicting on validation.csv (12,000 loads)...')
final_val_preds = model_full.predict(val_feat[features])
val_out = val_df[['load_id']].copy()
val_out['predicted_rate'] = final_val_preds
val_out.to_csv('output/validation_predictions.csv', index=False)

print('Predicting on december-chart-inputs.csv (31 rows)...')
dec_preds = model_full.predict(dec_feat[features])
dec_out = dec_df.copy()
dec_out['predicted_rate'] = dec_preds
dec_out.to_csv('output/december-chart-inputs.csv', index=False)

print('All predictions saved successfully in output/ directory!')